In [2]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

# 准备文档chunk

with open('../day07/resources/评估.md','r',encoding='utf-8') as file:
    # 读到文档内容
    doc_content = file.read()

# 创建切片对象
markdown_splitter = MarkdownHeaderTextSplitter(
     headers_to_split_on=[
        # 按照标题切分
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
    ],
    # 设置是否需要携带标题
    strip_headers=False
)

# 切片
chunks = markdown_splitter.split_text(doc_content)

In [4]:
from langchain_ollama import OllamaEmbeddings
# 创建向量模型,我们今天使用ollama

ollama_embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b", dimensions=1024)

In [5]:
# 初始化向量数据库客户端对象
from pymilvus import MilvusClient
from app.core.config import settings

# 初始化客户端
milvus_client = MilvusClient(uri=settings.rag.milvus_url)

# Collection名称:集合,指的就是表名字
collection_name = "my_collection_1"

In [7]:
# 利用客户端插入数据
# 对文档片段手动向量化
vectors = ollama_embeddings.embed_documents(
    [
        # 将内容全部向量化
        doc.page_content
        for doc in chunks
    ]
)

# 遍历出新数据,插入数据库
data = [
    # 向量数据库需要插入字典
    {
        "id": index+1,
        "content": doc.page_content,
        "h2": doc.metadata.get("Header 2", ""),
        "dense": vectors[index],  # 稠密向量字段
    }
    for index,doc in enumerate(chunks)
]
# 写入
res = milvus_client.upsert(collection_name,data)

In [8]:
# 删除和更新文档
copy_doc1 = data[0]
# 修改标题
copy_doc1['h2'] = '我是被修改的标题'

copy_doc2 = data[-1]
copy_doc2['id'] = 100

# 写入
result = milvus_client.upsert(collection_name,[copy_doc1,copy_doc2])

In [9]:
# 删除某一条数据
result = milvus_client.delete(collection_name,[1,100])
print(result)

{'delete_count': 2}
